# سبق 08 - کثیر ایجنٹ ڈیزائن پیٹرن


## سیٹ اپ


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity --quiet

import os
import asyncio

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## کیوں ملٹی ایجنٹ سسٹمز؟

حقیقی دنیا کے کام جیسے کہ سفر کی منصوبہ بندی میں مختلف قسم کی مہارتیں شامل ہوتی ہیں — لاجسٹکس، مقامی معلومات، بجٹ سازی، اور بہت کچھ۔ ایک واحد ایجنٹ جو سب کچھ سنبھالنے کی کوشش کرتا ہے جلد ہی ناقابلِ انتظام ہو جاتا ہے۔

ملٹی ایجنٹ سسٹمز اس مسئلے کو **مہارت کی تخصیص** کے ذریعے حل کرتے ہیں: ہر ایجنٹ ایک خاص شعبے پر توجہ مرکوز کرتا ہے، جو عمومی ایجنٹ کے مقابلے میں زیادہ معیاری نتائج فراہم کرتا ہے۔ یہ **قابلیت توسیع** کو بھی بہتر بناتے ہیں — آپ نئے ایجنٹس (جیسے کہ فلائٹ اسپیشلسٹ، ریسٹورنٹ نقاد) بغیر موجودہ ورک فلو کو دوبارہ لکھے شامل کر سکتے ہیں۔ ایجنٹس ایک منظم پائپ لائن کے ذریعے آپس میں جُڑے ہوتے ہیں، جہاں ایک سے دوسرے کو سیاق و سباق منتقل کیا جاتا ہے۔


## ماہر ایجنٹس کی تخلیق


In [ ]:
planner_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## سلسلہ وار ورک فلو کی تعمیر

`WorkflowBuilder` آپ کو ایجنٹس کو ایک سمت دار گراف میں جوڑنے دیتا ہے۔ یہاں ہم ایک سادہ دو قدمی پائپ لائن بناتے ہیں: **TravelPlanner** سفر کا منصوبہ تیار کرتا ہے، پھر **TravelConcierge** اس کا جائزہ لیتا ہے اور اسے بہتر بناتا ہے۔


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## ورک فلو میں مزید ایجنٹس شامل کرنا

ملٹی ایجنٹ پیٹرن کا سب سے بڑا فائدہ یہ ہے کہ اسے بڑھانا کتنا آسان ہے۔ نیچے ہم ایک **BudgetReviewer** ایجنٹ شامل کرتے ہیں جو منصوبے کو مسافر کے بجٹ کے مطابق چیک کرتا ہے، ایسی اشیاء کو نشان زد کرتا ہے جو اخراجات کو حد سے تجاوز کروا سکتی ہیں، اور پیسے بچانے والے متبادل تجویز کرتا ہے۔ ورک فلو اب تسلسل میں تین ایجنٹس چلتا ہے:

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = await provider.create_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## خلاصہ

اس درس میں آپ نے سیکھا کہ کیسے:

1. **خصوصی ایجنٹس بنائیں** — ہر ایک کا ایک مرکز کردار ہو (منصوبہ بندی، کنسیئرج، بجٹ کا جائزہ)۔
2. **ایجنٹس کو متسلسل ورک فلو میں جوڑیں** `WorkflowBuilder` اور `add_edge` کا استعمال کرتے ہوئے۔
3. **کئی ایجنٹوں کے پائپ لائن سے آؤٹ پٹ کو اسٹریم کریں**، اور یہ ٹریک کریں کہ کون سا ایجنٹ بول رہا ہے۔
4. **ورک فلو کو بڑھائیں** نئے ایجنٹس شامل کرکے بغیر موجودہ ایجنٹس میں تبدیلی کیے۔

کئی ایجنٹوں کے ڈیزائن پیٹرن ہر ایجنٹ کو سادہ رکھتا ہے جبکہ زیادہ بھرپور، زیادہ اچھی طرح سے جائزہ لیا گیا نتیجہ پیدا کرتا ہے جو کہ کوئی واحد ایجنٹ اکیلا حاصل نہیں کر سکتا۔


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**مثبتی دستبرداری**:  
یہ دستاویز AI ترجمہ سروس [Co-op Translator](https://github.com/Azure/co-op-translator) استعمال کرتے ہوئے ترجمہ کی گئی ہے۔ اگرچہ ہم درستگی کے لیے کوشاں ہیں، لیکن براہ کرم آگاہ رہیں کہ خودکار تراجم میں غلطیاں یا نقص ہو سکتے ہیں۔ اصل دستاویز اپنی مادری زبان میں ہی مستند ماخذ سمجھی جائے۔ اہم معلومات کے لیے پیشہ ور انسانی ترجمہ کی سفارش کی جاتی ہے۔ اس ترجمہ کے استعمال سے ہونے والی کسی بھی غلط فہمی یا غلط تعبیر کی ذمہ داری ہم پر نہیں ہوگی۔
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
